In [ ]:
import numpy as np, matplotlib.pyplot as plt, tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
np.random.seed(11); tf.random.set_seed(11)
# CIFAR-10 is the specified permitted substitute for the six waste classes
(x,y),(xt,yt)=tf.keras.datasets.cifar10.load_data(); y=y.ravel(); yt=yt.ravel()
x=x.astype("float32")/255.; xt=xt.astype("float32")/255.
x_train,x_val,y_train,y_val=train_test_split(x[:30000],y[:30000],test_size=.2,random_state=11,stratify=y[:30000])
x_test, y_test=xt[:5000],yt[:5000]
names=["airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck"]
model=tf.keras.Sequential([tf.keras.layers.Input((32,32,3)),tf.keras.layers.Conv2D(32,3,padding="same",activation="relu"),tf.keras.layers.MaxPooling2D(),tf.keras.layers.Conv2D(64,3,padding="same",activation="relu"),tf.keras.layers.MaxPooling2D(),tf.keras.layers.Flatten(),tf.keras.layers.Dense(128,activation="relu"),tf.keras.layers.Dense(10,activation="softmax")])
model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])
h=model.fit(x_train,y_train,validation_data=(x_val,y_val),epochs=8,batch_size=64,verbose=1)
loss,acc=model.evaluate(x_test,y_test,verbose=0); pred=model.predict(x_test,verbose=0).argmax(1)
print("Test loss:",round(loss,4),"Test accuracy:",round(acc,4));print(classification_report(y_test,pred,target_names=names,zero_division=0))
fig,ax=plt.subplots(1,2,figsize=(11,4));ax[0].plot(h.history["accuracy"],label="Train");ax[0].plot(h.history["val_accuracy"],label="Validation");ax[0].set_title("Accuracy");ax[1].plot(h.history["loss"],label="Train");ax[1].plot(h.history["val_loss"],label="Validation");ax[1].set_title("Loss")
for a in ax:a.legend();a.grid()
plt.tight_layout();plt.show()
ConfusionMatrixDisplay.from_predictions(y_test,pred,display_labels=names,xticks_rotation=45);plt.tight_layout();plt.show()
wrong=np.where(pred!=y_test)[0][:12]; fig,ax=plt.subplots(3,4,figsize=(10,7))
for a,i in zip(ax.ravel(),wrong):a.imshow(x_test[i]);a.set_title(f"T:{names[y_test[i]]}\nP:{names[pred[i]]}");a.axis("off")
plt.tight_layout();plt.show()
fm=tf.keras.Model(model.input,model.layers[0].output).predict(x_test[:1],verbose=0)
fig,ax=plt.subplots(4,4,figsize=(8,8))
for i,a in enumerate(ax.ravel()):a.imshow(fm[0,:,:,i],cmap="viridis");a.axis("off")
plt.suptitle("First convolution layer feature maps");plt.tight_layout();plt.show()